# Урок 7.3. CrewAI: та же команда на фреймворке

В уроке 7.2 мы построили оркестратор своими руками — 150 строк, полный контроль.
Сегодня соберём ту же команду на **CrewAI** — самом популярном фреймворке для
мульти-агентных систем — и подключим его к нашей локальной Ollama. Цель не
«выучить CrewAI», а честно понять, **что фреймворк делает за вас, что забирает
у вас — и почём**.

🎯 **Цели урока:**

- Собрать конвейер «аналитик → разработчик» на Agent/Task/Crew.
- Подключить CrewAI к локальной qwen2.5:3b через Ollama.
- Увидеть скрытую обвязку фреймворка и замерить её цену в токенах.
- Прикрутить к результату crew наш судья-тесты из 7.2 — и понять, почему
  фреймворк не отменяет собственную приёмку.
- Составить честную таблицу «свой оркестратор vs фреймворк».

**Что понадобится:** Ollama с qwen2.5:3b и отдельное окружение с `crewai`
(см. README модуля: `python -m venv .venv` + `pip install -r requirements.txt` —
CrewAI тянет несколько десятков пакетов, лучше держать его подальше от основного
окружения).

## 1. Зачем фреймворк после 150 строк

Что CrewAI даёт из коробки:

- декларативные **роли** и **задачи** — вместо ручной сборки промптов;
- готовые процессы: `sequential` (наш конвейер) и `hierarchical` (supervisor из 7.1);
- передачу контекста между задачами (наши `Message` и журнал);
- инструменты, память, интеграции — целая экосистема.

Что фреймворк забирает: **контроль над промптами** (он собирает их сам, на
английском, по своим шаблонам), **судью** (никакой встроенной проверки выходов
кодом) и **прозрачность цены** (обвязка ест токены незаметно).

И две вещи, которые стоит знать до `pip install`:

1. CrewAI по умолчанию собирает **телеметрию** и предлагает облачный трейсинг.
   Мы всё выключаем переменными окружения — привычка из урока 4.3: сначала
   узнай, что инструмент отправляет наружу, потом запускай.
2. Фреймворк развивается стремительно: код ниже написан под crewai 1.15.x
   (закреплено в requirements.txt модуля). Через полгода имена аргументов могут
   сдвинуться — принципы не сдвинутся.

In [1]:
import os

# телеметрия и облачный трейсинг - выключить ДО импорта crewai
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

from crewai import Agent, Crew, LLM, Process, Task

# наша локальная модель: провайдер "ollama/" + имя тега, адрес - 127.0.0.1 (урок 3.2)
llm = LLM(
    model="ollama/qwen2.5:3b",
    base_url="http://127.0.0.1:11434",
    temperature=0,
    max_tokens=700,          # num_predict: стоп-кран на каждый вызов (урок 5.6)
)
print("LLM настроена:", llm.model)

LLM настроена: qwen2.5:3b


## 2. Agent: роль в трёх полях

`Agent` в CrewAI — это наши системные промпты из 7.2, разложенные по полочкам:
`role` (кто), `goal` (что для него успех), `backstory` (контекст роли). Из этих
трёх полей фреймворк соберёт **свой** системный промпт — английский, по
внутреннему шаблону («You are {role}. {backstory} Your personal goal is:
{goal}...»). Наши русские тексты окажутся внутри английской обвязки — для
большой модели это мелочь, для 3B — источник сюрпризов (следим по ходу урока).

`allow_delegation=False` — важный выключатель: иначе агенты получают инструмент
«делегируй коллеге», и на маленькой модели это лотерея с бесконечными
перепасовками.

In [2]:
analyst = Agent(
    role="Аналитик",
    goal="Превращать задачи в короткие точные спецификации",
    backstory="Ты работаешь в хостинге «Пингвин.Хост» и пишешь спецификации для разработчиков.",
    llm=llm,
    verbose=True,             # показывать работу агента - нам это нужно для разбора
    allow_delegation=False,   # без перепасовок: у нас конвейер, а не базар
)
developer = Agent(
    role="Python-разработчик",
    goal="Писать рабочий код строго по спецификации",
    backstory="Ты пишешь чистый Python-код по спецификациям аналитика.",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)
print("агенты готовы")

агенты готовы


## 3. Task: задача с надеждой на выход

`Task` — единица работы: `description` (что делать), `expected_output` (каким
должен быть результат) и `context` (выходы каких задач передать на вход — это
наши сообщения между ролями).

Подвох, который важно увидеть сразу: **`expected_output` — это не валидация.**
Фреймворк просто вставит его в промпт как пожелание. Никто не проверит, что
разработчик действительно вернул «один блок кода» — проверять будем мы, кодом,
как в 7.2.

In [3]:
TASK_TEXT = """Функция Python normalize_phone(text: str) -> str: приводит российский
номер телефона к каноничному виду - строка из 11 цифр БЕЗ разделителей,
начинается с 7. Вход бывает в разнобой: "+7 (912) 345-67-89", "8-912-345-67-89",
с пробелами. Номер, начинающийся с 8, - это тот же номер с кодом 7.
Если номер не получается корректным - подними ValueError."""

spec_task = Task(
    description=(f"Задача:\n{TASK_TEXT}\n\n"
                 "Выпиши 4-6 требований и 3-4 крайних случая списками. НЕ пиши код."),
    expected_output="Нумерованный список требований и крайних случаев, без кода.",
    agent=analyst,
)
code_task = Task(
    description=("Напиши функцию normalize_phone по спецификации аналитика. "
                 "Верни ТОЛЬКО код одним блоком ```python```."),
    expected_output="Один блок кода ```python``` с функцией normalize_phone.",
    agent=developer,
    context=[spec_task],      # выход аналитика -> вход разработчика
)
print("задачи описаны")

задачи описаны


## 4. Crew: собрать и запустить

`Crew` = агенты + задачи + процесс. `Process.sequential` — наш конвейер:
задачи идут по порядку, контекст передаётся по стрелкам `context`.

In [4]:
import time

crew = Crew(
    agents=[analyst, developer],
    tasks=[spec_task, code_task],
    process=Process.sequential,
    verbose=True,
)

started = time.perf_counter()
# в Jupyter уже крутится event loop, поэтому синхронный crew.kickoff() падает
# с RuntimeError - в ноутбуках нужен await-вариант (в обычном скрипте: kickoff())
result = await crew.kickoff_async()
elapsed = time.perf_counter() - started

print("\n" + "=" * 60)
print(f"за {elapsed:.0f} c | {result.token_usage.successful_requests} запросов | "
      f"{result.token_usage.prompt_tokens} ток. промптов + "
      f"{result.token_usage.completion_tokens} ток. ответов")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7df48bcb-6a22-4d77-b64f-0c150ccd3d92                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Задача:                                                                                                  │
│  Функция Python normalize_phone(text: str) -> str: приводит российский                                          │
│  номер телефона к каноничному виду - строка из 11 цифр БЕЗ разделителей,                                        │
│  начинается с 7. Вход бывает в разнобой: "+7 (912) 345-67-89", "8-912-345-67-89",                               │
│  с пробелами. Номер, начинающийся с 8, - это тот же номер с кодом 7.                                            │
│  Если номер не получается корректным - подними ValueError.                                                      │
│                                                                                                                 │
│  Выпиши 4-6 требований и 3-4 крайних случая списками. НЕ пиши код.                                              │
│  ID: 59ade4d6-13bb-487f-98d5-13bc861bc753                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Аналитик                                                                                                │
│                                                                                                                 │
│  Task: Задача:                                                                                                  │
│  Функция Python normalize_phone(text: str) -> str: приводит российский                                          │
│  номер телефона к каноничному виду - строка из 11 цифр БЕЗ разделителей,                                        │
│  начинается с 7. Вход бывает в разнобой: "+7 (912) 345-67-89", "8-912-345-67-89",                               │
│  с пробелами. Номер, начинающийся с 8, - это тот же номер с кодом 7.                                            │
│  Если номер не получается корректным - подними ValueError.                                                      │
│                                                                                                                 │
│  Выпиши 4-6 требований и 3-4 крайних случая списками. НЕ пиши код.                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Аналитик                                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. Функция должна принимать строку в качестве входных данных и возвращать строку из 11 цифр без разделителей,  │
│  начинающуюся с 7.                                                                                              │
│  2. Входные данные могут содержать символы "+", " ", "-", и могут быть в разнобой.                              │
│  3. Если входная строка начинается с "+7", то она должна быть преобразована в строку из 11 цифр, начинающуюся   │
│  с 7.                                                                                                           │
│  4. Если входная строка начинается с "8", то она должна быть преобразована в строку из 11 цифр, начинающуюся с  │
│  7.                                                                                                             │
│  5. Если номер не может быть корректно преобразован в строку из 11 цифр, то должна быть поднята ValueError.     │
│  6. Если входная строка содержит символы, не входящие в перечисленные выше, то она должна быть обработана как   │
│  некорректный вход.                                                                                             │
│                                                                                                                 │
│  Крайние случаи:                                                                                                │
│                                                                                                                 │
│  - Крайний случай: входная строка "+7 (912) 345-67-89". Функция должна вернуть "79123456789".                   │
│  - Крайний случай: входная строка "8-912-345-67-89". Функция должна вернуть "79123456789".                      │
│  - Крайний случай: входная строка "7912 345 67 89". Функция должна вернуть "79123456789".                       │
│  - Крайний случай: входная строка "79123456789". Функция должна вернуть "79123456789".                          │
│  - Крайний случай: входная строка "7912 345 67 89". Функция должна вернуть "79123456789".                       │
│  - Крайний случай: входная строка "79123456789". Функция должна вернуть "79123456789".                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Python-разработчик                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  def normalize_phone(phone_number):                                                                             │
│      # Remove spaces and replace hyphens with spaces                                                            │
│      phone_number = phone_number.replace(" ", "").replace("-", " ")                                             │
│                                                                                                                 │
│      # Check if the number starts with "+7" or "8"                                                              │
│      if phone_number.startswith("+7") or phone_number.startswith("8"):                                          │
│          phone_number = "7" + phone_number[2:]                                                                  │
│                                                                                                                 │
│      # Check if the number starts with "7"                                                                      │
│      if phone_number.startswith("7"):                                                                           │
│          phone_number = phone_number[1:]                                                                        │
│                                                                                                                 │
│      # Validate the phone number format                                                                         │
│      if not phone_number.startswith("7") or not phone_number[1:].isdigit() or len(phone_number) != 11:          │
│          raise ValueError("Invalid phone number format")                                                        │
│                                                                                                                 │
│      return phone_number                                                                                        │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Напиши функцию normalize_phone по спецификации аналитика. Верни ТОЛЬКО код одним блоком ```python```.    │
│  Agent: Python-разработчик                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7df48bcb-6a22-4d77-b64f-0c150ccd3d92                                                                       │
│  Final Output: ```python                                                                                        │
│  def normalize_phone(phone_number):                                                                             │
│      # Remove spaces and replace hyphens with spaces                                                            │
│      phone_number = phone_number.replace(" ", "").replace("-", " ")                                             │
│                                                                                                                 │
│      # Check if the number starts with "+7" or "8"                                                              │
│      if phone_number.startswith("+7") or phone_number.startswith("8"):                                          │
│          phone_number = "7" + phone_number[2:]                                                                  │
│                                                                                                                 │
│      # Check if the number starts with "7"                                                                      │
│      if phone_number.startswith("7"):                                                                           │
│          phone_number = phone_number[1:]                                                                        │
│                                                                                                                 │
│      # Validate the phone number format                                                                         │
│      if not phone_number.startswith("7") or not phone_number[1:].isdigit() or len(phone_number) != 11:          │
│          raise ValueError("Invalid phone number format")                                                        │
│                                                                                                                 │
│      return phone_number                                                                                        │
│  ```                                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


за 13 c | 4 запросов | 1870 ток. промптов + 1278 ток. ответов


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Мой прогон: конвейер отработал за ~13 секунд, и `token_usage` рассказывает
больше, чем любые панели:

- **4 запроса на 2 задачи.** Фреймворк делает свои служебные проходы поверх
  наших: у каждой задачи есть внутренний цикл агента со сведением ответа
  к формату. В нашем оркестраторе из 7.2 вызовов было ровно столько, сколько
  ролей в конвейере.
- **1870 токенов промптов** — против ~700, которые стоил тот же конвейер
  «аналитик → разработчик» голыми вызовами в уроке 7.1. Разница — английская
  обвязка ролей и задач, помноженная на 4 запроса.
- В интерактивном Jupyter `verbose=True` рисует панели с ходом каждой задачи —
  загляните туда один раз, чтобы увидеть скрытые промпты фреймворка своими
  глазами («You are Аналитик... This is the expected criteria...» — наши русские
  тексты плавают в английском шаблоне).

Пока всё выглядит удобным: 40 строк вместо 150. Теперь главный вопрос
урока 7.2 — а код-то рабочий?

## 5. Судья остаётся нашим

Crew отработала и вернула «результат». В уроке 7.2 мы уже знаем цену слову
«результат» без приёмки. Прикручиваем **тот же харнес** — он ничего не знает
про CrewAI, ему всё равно, кто написал код:

In [5]:
# судья из урока 7.2 - без изменений
import json
import re
import subprocess
import sys
import tempfile
from pathlib import Path

def extract_code(text: str) -> str | None:
    match = re.search(r"```(?:python)?\s*(.+?)```", text, re.DOTALL)
    return match.group(1).strip() if match else None

HARNESS = """
import json
results = []
{code}
for expr, expected in {tests!r}:
    try:
        got = eval(expr)
        results.append({{"case": expr, "ok": got == expected, "got": repr(got), "expected": expected}})
    except Exception as exc:
        results.append({{"case": expr, "ok": False, "got": f"{{type(exc).__name__}}: {{exc}}", "expected": expected}})
for expr in {errors!r}:
    try:
        eval(expr)
        results.append({{"case": expr + " => ValueError", "ok": False, "got": "исключения не было", "expected": "ValueError"}})
    except ValueError:
        results.append({{"case": expr + " => ValueError", "ok": True, "got": "ValueError", "expected": "ValueError"}})
    except Exception as exc:
        results.append({{"case": expr + " => ValueError", "ok": False, "got": f"{{type(exc).__name__}}", "expected": "ValueError"}})
print(json.dumps(results, ensure_ascii=False))
"""

def run_tests(code_str, tests, error_cases):
    source = HARNESS.format(code=code_str, tests=tests, errors=error_cases)
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False,
                                     encoding="utf-8") as f:
        f.write(source)
        path = f.name
    try:
        proc = subprocess.run([sys.executable, "-X", "utf8", path],
                              capture_output=True, text=True, encoding="utf-8",
                              timeout=10)
        if proc.returncode != 0:
            return [{"case": "<загрузка кода>", "ok": False,
                     "got": proc.stderr.strip()[-200:], "expected": "импорт без ошибок"}]
        return json.loads(proc.stdout)
    except subprocess.TimeoutExpired:
        return [{"case": "<выполнение>", "ok": False, "got": "timeout 10 c",
                 "expected": "завершение"}]
    finally:
        Path(path).unlink(missing_ok=True)

TESTS = [
    ('normalize_phone("+7 (912) 345-67-89")', "79123456789"),
    ('normalize_phone("8-912-345-67-89")', "79123456789"),
    ('normalize_phone("79123456789")', "79123456789"),
    ('normalize_phone("8 (495) 123 45 67")', "74951234567"),
]
ERROR_CASES = ['normalize_phone("12345")', 'normalize_phone("нет телефона")']

code_str = extract_code(str(result)) or str(result)
verdicts = run_tests(code_str, TESTS, ERROR_CASES)
for r in verdicts:
    print(f"[{'PASS' if r['ok'] else 'FAIL'}] {r['case']} -> {r['got']} (ожидалось {r['expected']})")
print(f"\nприёмка: {sum(r['ok'] for r in verdicts)}/{len(verdicts)}")

[FAIL] normalize_phone("+7 (912) 345-67-89") -> ValueError: Invalid phone number format (ожидалось 79123456789)
[FAIL] normalize_phone("8-912-345-67-89") -> ValueError: Invalid phone number format (ожидалось 79123456789)
[FAIL] normalize_phone("79123456789") -> ValueError: Invalid phone number format (ожидалось 79123456789)
[FAIL] normalize_phone("8 (495) 123 45 67") -> ValueError: Invalid phone number format (ожидалось 74951234567)
[PASS] normalize_phone("12345") => ValueError -> ValueError (ожидалось ValueError)
[PASS] normalize_phone("нет телефона") => ValueError -> ValueError (ожидалось ValueError)

приёмка: 2/6


**2/6 в моём прогоне** — прошли только кейсы про ValueError: на всех настоящих
номерах код падает с «Invalid phone number format». Панель фреймворка при этом
сияла зелёным «Task Completed» — потому что «completed» значит «модель ответила»,
а не «код работает».

Это не претензия к CrewAI: судить выходы — не его работа, и он честно этого
не обещал. Претензия возникает к архитектору, который решил, что фреймворк
эту работу сделает. Приёмка, фикс-цикл, эскалация, лимиты из урока 7.2 — всё это
надо строить **вокруг** crew (упражнение 2 предлагает ровно это): фреймворк
оказывается деталью внутри вашего оркестратора, а не наоборот.

## 6. Цена обвязки — в цифрах

Отдельный замер вне урока: задача «скажи слово готово» через Crew из одного
агента стоила **86 токенов промпта** (у голого вызова `/api/chat` — около 20).
Разница — та самая английская обвязка: шаблон роли, правила формата ответа,
служебные инструкции. На каждый вызов, всегда.

Для нашего конвейера это терпимо (сотни лишних токенов на прогон). Важно другое:
**вы не управляете этим промптом.** Если 3B-модель начнёт отвечать по-английски
или ломать формат «Final Answer» (внутренний протокол CrewAI) — чинить придётся
не промптом, которого вы не видите, а обходными путями: жёстче `description`,
ниже температура, больше модель.

| | наш оркестратор (7.2) | CrewAI |
|---|---|---|
| строк кода до первого прогона | ~150 | ~40 |
| контроль промптов | полный | role/goal/backstory, остальное — шаблон |
| судья кодом | встроили сами | нет — прикручивайте сами |
| фикс-цикл, эскалация, лимиты | в коде, любые | стройте поверх (retry/flows) |
| телеметрия каждого вызова | своя, полная | `token_usage` суммарно |
| инструменты/память/интеграции | руками | из коробки, много |
| supervisor/делегирование | писать самому | `Process.hierarchical`, флаг |
| зависимости | httpx | десятки пакетов |
| поведение на 3B | предсказуемое (свои промпты) | англ. обвязка добавляет сюрпризов |

Правило выбора — то же, что всегда в этом курсе: **фреймворк берут за
экосистему** (инструменты, иерархия, интеграции, команда, которой нужен общий
язык), **а не за «так проще начать»**. Начать проще со 150 строк, которые вы
понимаете до последней строчки.

## ⚠️ Частые ошибки

1. **`expected_output` как валидация.** Это пожелание в промпте, не проверка.
   Выход задачи проверяет только ваш код (харнес, схема, грамматика 5.4).
2. **Оставить телеметрию/трейсинг по умолчанию.** Локальная LLM ради приватности +
   фреймворк, звонящий домой, — так себе комбинация. Выключайте до импорта.
3. **`allow_delegation=True` на маленькой модели.** Агенты начинают вежливо
   перекидывать задачу друг другу; бюджет тает, результата нет.
4. **Не пинить версию фреймворка.** CrewAI ломает API быстрее, чем вы дочитаете
   его changelog. В requirements — точная версия, обновление — осознанное.
5. **Судить о работе по красивым панелям verbose.** Панель «Task Completed» —
   это не приёмка (наша приёмка выше это только что показала).
6. **Тащить фреймворк в задачу на один конвейер.** Sequential из двух задач —
   это 20 строк своего кода без 60 новых зависимостей.
7. **`crew.kickoff()` в Jupyter.** Внутри ноутбука уже работает event loop, и
   синхронный kickoff падает с RuntimeError. В ноутбуках — `await
   crew.kickoff_async()`, в скриптах — обычный `kickoff()`.

## Упражнения

1. **Третья роль.** Добавьте в crew задачу-ревьюера (агент «Код-ревьюер», context
   от `code_task`): пусть найдёт проблемы в коде. Сравните его замечания с
   отчётом нашего харнеса: что поймал LLM-ревьюер, а что — только тесты?

2. **Фикс-цикл вокруг crew.** CrewAI не дообрабатывает по красным тестам — но
   ничто не мешает обернуть `kickoff` нашим циклом из 7.2: прогнали crew →
   тесты → если красные, соберите НОВУЮ crew, где в description разработчика
   вшит отчёт тестов. Реализуйте и сравните сходимость с уроком 7.2.

3. **Честный замер обвязки.** Повторите конвейер «аналитик → разработчик»
   голыми вызовами `/api/chat` (код из 7.2) и через CrewAI, с одинаковыми
   текстами ролей. Сравните `token_usage` crew с суммой ваших счётчиков:
   сколько токенов стоит комфорт?

**Решение 1 (эскиз).**

```python
reviewer = Agent(role="Код-ревьюер", goal="Находить дефекты и риски в коде",
                 backstory="Ты придирчиво проверяешь чужой Python-код.",
                 llm=llm, verbose=True, allow_delegation=False)
review_task = Task(
    description="Найди проблемы в коде разработчика: краевые случаи, ошибки, несоответствия задаче.",
    expected_output="Нумерованный список проблем с пояснениями.",
    agent=reviewer, context=[code_task])
```

Ожидаемая картина — как в 7.1: ревьюер напишет правдоподобные общие замечания
(«не хватает обработки пустой строки»), но конкретный падающий кейс с got/expected
находят только тесты. LLM-ревью дополняет судью-код, не заменяет.

**Решение 2 (эскиз).** Внешний цикл:

```python
report = ""
for attempt in range(3):
    code_task = Task(description=BASE_DESCRIPTION + (f"\n\nОтчёт тестов прошлой попытки:\n{report}" if report else ""),
                     expected_output="...", agent=developer, context=[spec_task])
    crew = Crew(agents=[analyst, developer], tasks=[spec_task, code_task],
                process=Process.sequential)
    code_str = extract_code(str(crew.kickoff())) or ""
    verdicts = run_tests(code_str, TESTS, ERROR_CASES)
    if all(r["ok"] for r in verdicts):
        break
    report = "\n".join(f"{r['case']} -> {r['got']} (ожидалось {r['expected']})"
                        for r in verdicts if not r["ok"])
```

Обратите внимание: оркестратором снова стал ваш код — фреймворк оказался внутри
цикла, а не снаружи. Это типичный финал взросления мульти-агентных систем.

**Решение 3** — сравните и время тоже: каждая лишняя сотня токенов промпта — это
prefill (урок 5.2), на CPU он заметен.

## Мини-квиз

**1. Чем `expected_output` в Task отличается от нашего `run_tests` из 7.2?**

<details><summary>Ответ</summary>

`expected_output` — текст, который фреймворк вставляет в промпт как пожелание:
модель может его выполнить, а может нет — никто не проверит. `run_tests` —
исполняемая проверка с вердиктом. Первое — надежда, второе — приёмка.
</details>

**2. Почему на маленькой модели `allow_delegation=True` опасен?**

<details><summary>Ответ</summary>

Делегирование — это инструмент, который модель вызывает по своему усмотрению
через служебный формат. 3B путается в формате и критериях «кому передать»,
получаются циклы перепасовок — токены горят, задача стоит. Плюс каждый передённый
контекст — это ещё сотни токенов обвязки.
</details>

**3. Crew вернула результат, панель «Task Completed» зелёная. Что мы знаем о качестве кода?**

<details><summary>Ответ</summary>

Ничего. «Completed» означает «модель что-то ответила по всем задачам». Наша
приёмка тестами в этом уроке приняла бы решение и без панелей — и именно она
показала реальный счёт.
</details>

**4. Зачем выключать телеметрию CrewAI, если мы и так работаем на локальной модели?**

<details><summary>Ответ</summary>

Телеметрия — это исходящие запросы самого фреймворка (статистика запусков,
трейсы по подписке). Локальность модели не выключает сетевую активность обвязки —
проверяйте и выключайте явно (урок 4.3: знай, что уходит наружу).
</details>

**5. В каких случаях CrewAI — правильный выбор, несмотря на потерю контроля?**

<details><summary>Ответ</summary>

Когда нужна его экосистема: готовые инструменты и интеграции, иерархический
процесс, память, командная разработка на общем словаре Agent/Task/Crew — и когда
модель достаточно большая, чтобы прощать английскую обвязку. Для конвейера из
двух-трёх ролей на локальной 3B свой оркестратор проще и предсказуемее.
</details>

## 🎓 Итоги

- CrewAI: `LLM` (ollama/тег + 127.0.0.1) → `Agent` (role/goal/backstory) →
  `Task` (description/expected_output/context) → `Crew` (sequential) → `kickoff`.
  Конвейер из 7.2 собрался в ~40 строк.
- Фреймворк оркеструет, но **не судит**: `expected_output` — пожелание, приёмка —
  ваш код. Наш харнес из 7.2 подключился без изменений.
- Обвязка стоит токенов (86 против ~20 на пустяковой задаче) и написана
  по-английски — на 3B это источник сюрпризов, управлять которым нельзя.
- Телеметрию фреймворков выключаем до импорта; версию — пиним.
- Выбор: экосистема и стандартные паттерны — фреймворк; контроль, судьи и
  лимиты — свой оркестратор. Часто финал — гибрид: crew внутри вашего цикла.

**Дальше — урок 7.4:** альтернативы: AutoGen/AG2 и LangGraph для мультиагентов,
сравнительная таблица и критерии выбора.

## 📚 Что почитать

- [Документация CrewAI](https://docs.crewai.com/) — Agent/Task/Crew/Flows
- [CrewAI + Ollama](https://docs.crewai.com/concepts/llms) — подключение локальных моделей
- [Anthropic: Building effective agents](https://www.anthropic.com/research/building-effective-agents) — контрапункт: когда фреймворк не нужен
